In [2]:
import pandas as pd
import numpy as np

def proses_data_bri(file_mentah, batch_id_input, tanggal_target, id_awal):
    print(f"Memproses file: '{file_mentah}'")
    print(f"Mencari data untuk Tanggal Submit: {tanggal_target} (Batch ID: {batch_id_input})...")

    df_raw = pd.read_excel(file_mentah, sheet_name='Sheet1', skiprows=2, engine='openpyxl')
    df_raw = df_raw.dropna(subset=['NOMOR POLIS'])

    df_raw['TANGGAL SUBMIT'] = pd.to_datetime(df_raw['TANGGAL SUBMIT']).dt.strftime('%Y-%m-%d')
    df_raw = df_raw[df_raw['TANGGAL SUBMIT'] == tanggal_target]

    if df_raw.empty:
        print(f"[PERINGATAN] Tidak ada data yang ditemukan untuk tanggal {tanggal_target}.")
        return None

    print(f"Ditemukan {len(df_raw)} baris data untuk tanggal {tanggal_target}.")

    kolom_mapping = {
        'NOMOR POLIS': 'polis_number',
        'NAMA PEMEGANG POLIS': 'owner',
        'NAMA TERTANGGUNG': 'insured',
        'JUP': 'up',
        'TANGGAL MULAI ASURANSI': 'effective_date',
        'NOMOR TELEPON AHLI WARIS': 'beneficiary_phone_number',
        'CATATAN PENGAJUAN INVESTIGASI': 'note_analys',
        'WILAYAH': 'province_id'
    }

    df_clean = df_raw.rename(columns=kolom_mapping)

    df_clean['polis_type_id'] = 1
    df_clean['type_id'] = 1
    df_clean['client_id'] = 3
    df_clean['batch_id'] = batch_id_input

    def tentukan_case_type(jenis_klaim):
        jenis_klaim_str = str(jenis_klaim).upper()
        if 'LIFE' in jenis_klaim_str or 'MENINGGAL' in jenis_klaim_str:
            return 2
        elif 'CASHLESS' in jenis_klaim_str:
            return 1
        elif 'REIMBURSEMENT' in jenis_klaim_str:
            return 3
        return 2

    df_clean['case_type_id'] = df_clean['JENIS KLAIM'].apply(tentukan_case_type)

    # ==========================================
    # LOGIKA BARU SESUAI FORMULA IFS ANDA
    # ==========================================
    def tentukan_claim_type(jenis_klaim):
        # Menghilangkan spasi ekstra di awal/akhir dan ubah ke huruf besar
        jenis_klaim_str = str(jenis_klaim).strip().upper()

        # Pengecekan persis seperti formula =IFS Excel Anda
        if jenis_klaim_str == "CREDIT LIFE" or jenis_klaim_str == "CREDIT LIFE MENINGGAL DUNIA BIASA":
            return 200
        elif jenis_klaim_str == "INDIVIDU CRITICAL ILLNES" or "CRITICAL ILLNES" in jenis_klaim_str:
            return 150
        elif jenis_klaim_str == "INDIVIDU MENINGGAL DUNIA BIASA" or jenis_klaim_str == "INDIVIDU":
            return 100

        # Fallback (Jaga-jaga jika ada input yang typo / beda spasi)
        if 'CREDIT LIFE' in jenis_klaim_str:
            return 200
        elif 'INDIVIDU' in jenis_klaim_str:
            return 100

        return np.nan

    df_clean['polis_claim_type_id'] = df_clean['JENIS KLAIM'].apply(tentukan_claim_type)
    # ==========================================

    df_clean['id'] = range(id_awal, id_awal + len(df_clean))

    kolom_database_lengkap = [
        'id', 'batch_id', 'polis_type_id', 'polis_claim_type_id', 'type_id', 'case_type_id',
        'client_id', 'polis_number', 'owner', 'spaj_number', 'identity_card_number',
        'place_of_birth', 'date_of_birth', 'insured', 'identity_card_insured_number',
        'passport_insured', 'place_of_birth_insured', 'date_of_birth_insured', 'gender_id',
        'gender_insured_id', 'marital_status_id', 'marital_status_insured_id',
        'biological_mother_name', 'beneficiary', 'beneficiary_phone_number', 'relationship_id',
        'up', 'premi', 'agen_id', 'work_id', 'effective_date', 'address', 'province_id',
        'city_id', 'district_id', 'village_id', 'address_residence', 'province_residence_id',
        'city_residence_id', 'district_residence_id', 'village_residence_id', 'address_work',
        'province_work_id', 'city_work_id', 'district_work_id', 'village_work_id',
        'medical_history', 'rider_cover_date', 'claim_no_reff', 'note_analys', 'revision_note',
        'status', 'special_attachment_status', 'admin_status', 'verifier_status', 'created_at',
        'updated_at'
    ]

    df_final = df_clean.reindex(columns=kolom_database_lengkap)
    df_final = df_final.fillna('')

    return df_final

#
#Set dulu sebelum running
if __name__ == "__main__":
    nama_file_mentah = 'REGISTER ONDESK CALL SJA 2026 (5).xlsx'
    id_batch_sekarang = 799
    tanggal_yang_mau_diambil = '2026-03-25'
    id_awal_di_database = 101741

    try:
        df_hasil = proses_data_bri(
            nama_file_mentah,
            id_batch_sekarang,
            tanggal_yang_mau_diambil,
            id_awal_di_database
        )

        if df_hasil is not None:
            nama_file_output = f'Data_Siap_Import_BRI_{tanggal_yang_mau_diambil}.xlsx'
            df_hasil.to_excel(nama_file_output, index=False, engine='openpyxl')

            print(f"\n[SUKSES] Proses selesai!")
            print(f"File output disimpan sebagai: {nama_file_output}")

    except FileNotFoundError:
        print(f"[ERROR] File '{nama_file_mentah}' tidak ditemukan.")
    except Exception as e:
        print(f"[ERROR] Terjadi kesalahan: {e}")

Memproses file: 'REGISTER ONDESK CALL SJA 2026 (5).xlsx'
Mencari data untuk Tanggal Submit: 2026-03-25 (Batch ID: 799)...
Ditemukan 100 baris data untuk tanggal 2026-03-25.

[SUKSES] Proses selesai!
File output disimpan sebagai: Data_Siap_Import_BRI_2026-03-25.xlsx


C:\Users\muham\AppData\Local\Temp\ipykernel_18608\152767350.py:95: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df_final = df_final.fillna('')
